In [1]:
!pip install -q transformers datasets seqeval

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
print("✅ Login done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ Login done


In [2]:
import json, os

BASE      = "/kaggle/input/datasets/brownsugar297/bangla-dialect-nlp"
TRAIN_NER = f"{BASE}/train_ner.jsonl"
VAL_NER   = f"{BASE}/val_ner.jsonl"
TEST_NER  = f"{BASE}/test_ner.jsonl"

EXPECTED_NER = {"train_ner.jsonl": 4645, "val_ner.jsonl": 671, "test_ner.jsonl": 673}

for path in [TRAIN_NER, VAL_NER, TEST_NER]:
    fname = os.path.basename(path)
    with open(path) as f:
        lines = f.readlines()
    assert len(lines) == EXPECTED_NER[fname], f"❌ {fname}: expected {EXPECTED_NER[fname]}, got {len(lines)}"
    print(f"✅ {fname}: {len(lines)} rows")

✅ train_ner.jsonl: 4645 rows
✅ val_ner.jsonl: 671 rows
✅ test_ner.jsonl: 673 rows


In [3]:
ENTITIES = ["REL","LOC","OBJ","FOOD","ORG","COL","ROLE","ANI","PER"]
label_list = ["O"] + [f"B-{e}" for e in ENTITIES] + [f"I-{e}" for e in ENTITIES]
label2id = {l:i for i,l in enumerate(label_list)}
id2label  = {i:l for l,i in label2id.items()}

assert len(label_list) == 19, f"Expected 19 labels, got {len(label_list)}"
print(f"✅ {len(label_list)} labels: {label_list}")


✅ 19 labels: ['O', 'B-REL', 'B-LOC', 'B-OBJ', 'B-FOOD', 'B-ORG', 'B-COL', 'B-ROLE', 'B-ANI', 'B-PER', 'I-REL', 'I-LOC', 'I-OBJ', 'I-FOOD', 'I-ORG', 'I-COL', 'I-ROLE', 'I-ANI', 'I-PER']


In [4]:
# ============================================================
# CELL 4 
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss for token classification.
    Handles padding via ignore_index=-100.
    gamma=2.0: standard value from Lin et al. 2017.
    alpha: per-class weights tensor of shape [num_labels].
           Computed from inverse class frequency, capped at 10.
    """
    def __init__(self, gamma=2.0, alpha=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, logits, labels):
        num_labels = logits.size(-1)
        logits_flat = logits.view(-1, num_labels)
        labels_flat = labels.view(-1)
        active = labels_flat != self.ignore_index
        logits_active = logits_flat[active]
        labels_active = labels_flat[active]
        ce_loss = F.cross_entropy(
            logits_active, labels_active,
            weight=self.alpha.to(logits.device) if self.alpha is not None else None,
            reduction="none"
        )
        pt = torch.exp(-ce_loss)
        focal = ((1 - pt) ** self.gamma) * ce_loss
        return focal.mean()

print("✅ FocalLoss class defined (alpha will be set in Cell 6b after downsampling)")

✅ FocalLoss class defined (alpha will be set in Cell 6b after downsampling)


In [5]:
# ============================================================
# CELL 5 — Focal Loss 
# ============================================================

from transformers import Trainer

class FocalLossTrainer(Trainer):
    """Drop-in Trainer replacement that uses Focal Loss instead of cross-entropy."""
    def __init__(self, focal_loss_fn, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = focal_loss_fn

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.focal_loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

print("✅ FocalLossTrainer defined")

✅ FocalLossTrainer defined


In [6]:
# CELL 6 — Load Raw Data + All-O Downsampling
import random
from datasets import Dataset

random.seed(42)
KEEP_ALL_O_RATIO = 0.35

raw_train_rows = [json.loads(l) for l in open(TRAIN_NER, encoding="utf-8")]
entity_rows    = [r for r in raw_train_rows if any(t != "O" for t in r["tags"])]
all_o_rows     = [r for r in raw_train_rows if all(t == "O" for t in r["tags"])]

random.shuffle(all_o_rows)

# ✅ FIXED: keeps 35% of FINAL dataset as all-O (not 35% of all_o_rows)
keep_n = int(len(entity_rows) * KEEP_ALL_O_RATIO / (1 - KEEP_ALL_O_RATIO))
keep_n = min(keep_n, len(all_o_rows))  # cap করা যদি pool ছোট হয়

downsampled_train = entity_rows + all_o_rows[:keep_n]
random.shuffle(downsampled_train)

raw_val_rows  = [json.loads(l) for l in open(VAL_NER,  encoding="utf-8")]
raw_test_rows = [json.loads(l) for l in open(TEST_NER, encoding="utf-8")]

actual_all_o_pct = keep_n / len(downsampled_train) * 100
print(f"Original train:    {len(raw_train_rows)} rows")
print(f"  Entity rows:     {len(entity_rows)}")
print(f"  All-O rows:      {len(all_o_rows)} → keeping {keep_n} ({actual_all_o_pct:.1f}%)")
print(f"Downsampled train: {len(downsampled_train)} rows")

Original train:    4645 rows
  Entity rows:     2091
  All-O rows:      2554 → keeping 1125 (35.0%)
Downsampled train: 3216 rows


In [7]:
from collections import Counter
import numpy as np

all_tags = [t for r in downsampled_train for t in r["tags"]]
counts   = Counter(all_tags)
total    = len(all_tags)

weights = []
for lab in label_list:
    freq = counts.get(lab, 1)
    w = total / (len(label_list) * freq)
    weights.append(w)

weights = list(np.sqrt(weights))          # sqrt smoothing
weights = [min(w, 3.0) for w in weights]  # cap 3.0
weights[label_list.index("O")] = 1.0      # O fixed

alpha = torch.tensor(weights, dtype=torch.float)
focal_loss_fn = FocalLoss(gamma=1.0, alpha=alpha)  # γ 2.0 → 1.0

print("✅ Weights (sqrt, max=3.0, O=1.0, γ=1.0):")
for l, w in zip(label_list, weights):
    print(f"  {l:<10} {w:.3f}")

✅ Weights (sqrt, max=3.0, O=1.0, γ=1.0):
  O          1.000
  B-REL      1.054
  B-LOC      1.425
  B-OBJ      1.471
  B-FOOD     1.600
  B-ORG      2.523
  B-COL      2.601
  B-ROLE     2.899
  B-ANI      3.000
  B-PER      3.000
  I-REL      2.865
  I-LOC      3.000
  I-OBJ      3.000
  I-FOOD     3.000
  I-ORG      3.000
  I-COL      3.000
  I-ROLE     3.000
  I-ANI      3.000
  I-PER      3.000


In [8]:
#cell 7

from transformers import AutoTokenizer, DataCollatorForTokenClassification
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

def build_tokenized_datasets(tokenizer, train_data, val_data, test_data):
    """
    Tokenize with each model's own tokenizer.
    CRITICAL: mBERT and BanglaBERT use different tokenizers.
    """
    def tokenize_and_align(examples):
        tok = tokenizer(
            examples["tokens"], truncation=True,
            is_split_into_words=True, max_length=64,
        )
        labs = []
        for i, lbl in enumerate(examples["tags"]):
            word_ids = tok.word_ids(batch_index=i)
            prev = None; label_ids = []
            for w in word_ids:
                if w is None:    label_ids.append(-100)
                elif w != prev:  label_ids.append(label2id[lbl[w]])
                else:            label_ids.append(-100)
                prev = w
            labs.append(label_ids)
        tok["labels"] = labs
        return tok

    return {
        split: Dataset.from_list(data).map(
            tokenize_and_align, batched=True,
            remove_columns=["dialect","tokens","tags"]
        )
        for split, data in [
            ("train",train_data),
            ("validation",val_data),
            ("test",test_data)
        ]
    }

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_labels = [[id2label[l] for l in lab if l!=-100] for lab in labels]
    true_preds  = [[id2label[p] for p,l in zip(pr,lab) if l!=-100]
                   for pr,lab in zip(preds,labels)]
    # All 9 entity classes included in macro — PER/ANI flagged in paper Limitations
    return {
        "f1_macro":        f1_score(true_labels, true_preds, average="macro"),
        "precision_macro": precision_score(true_labels, true_preds, average="macro"),
        "recall_macro":    recall_score(true_labels, true_preds, average="macro"),
    }

print("✅ Tokenize function + metrics ready")

✅ Tokenize function + metrics ready


In [9]:
from transformers import AutoModelForTokenClassification, TrainingArguments, EarlyStoppingCallback, AutoTokenizer, DataCollatorForTokenClassification, Trainer
import json as _json

# mBERT uses its OWN tokenizer
tokenizer_mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
collator_mbert  = DataCollatorForTokenClassification(tokenizer_mbert)
tok_mbert = build_tokenized_datasets(tokenizer_mbert, downsampled_train, raw_val_rows, raw_test_rows)

model_mbert = AutoModelForTokenClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=len(label_list), id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)

trainer_mbert = Trainer(
    model=model_mbert,
    args=TrainingArguments(
        output_dir="/kaggle/working/outputs_mbert",
        num_train_epochs            = 10,
        per_device_train_batch_size = 16,
        learning_rate               = 2e-5,
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,

        eval_strategy="epoch",
        save_strategy="epoch",
        save_only_model=True,   # ← এটা add করো
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        seed=42,
        report_to="none",
    ),
    train_dataset=tok_mbert["train"],
    eval_dataset=tok_mbert["validation"],
    processing_class=tokenizer_mbert,
    data_collator=collator_mbert,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_mbert.train()

mbert_test_results = trainer_mbert.evaluate(tok_mbert["test"])
print(f"✅ Baseline 1 — mBERT + CrossEntropy | Test Macro F1: {mbert_test_results['eval_f1_macro']:.4f}")

with open("/kaggle/working/mbert_ner_results.json", "w") as f:
    _json.dump(mbert_test_results, f)

print("✅ mBERT results saved")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3216 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

✅ Baseline 1 — mBERT + CrossEntropy | Test Macro F1: 0.6275
✅ mBERT results saved


In [10]:
#CELL 8b — Baseline 2: BanglaBERT + CrossEntropy (BB tokenizer)
#============================================================
# BanglaBERT uses its OWN tokenizer
tokenizer_bb = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")
collator_bb  = DataCollatorForTokenClassification(tokenizer_bb)
tok_bb = build_tokenized_datasets(tokenizer_bb, downsampled_train, raw_val_rows, raw_test_rows)

model_bb_ce = AutoModelForTokenClassification.from_pretrained(
    "csebuetnlp/banglabert",
    num_labels=len(label_list), id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)

trainer_bb_ce = Trainer(
    model=model_bb_ce,
    args=TrainingArguments(
        output_dir="/kaggle/working/outputs_ner_bb_ce",
        num_train_epochs            = 10,
        per_device_train_batch_size = 16,
        learning_rate               = 2e-5,
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_only_model=True,   # ← এটা add করো
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        seed=42, report_to="none",
    ),
    train_dataset=tok_bb["train"],
    eval_dataset=tok_bb["validation"],
    processing_class=tokenizer_bb,
    data_collator=collator_bb,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_bb_ce.train()
bb_ce_results = trainer_bb_ce.evaluate(tok_bb["test"])
print(f"✅ Baseline 2 — BanglaBERT + CrossEntropy | Test Macro F1: {bb_ce_results['eval_f1_macro']:.4f}")

with open("/kaggle/working/bb_ce_ner_results.json", "w") as f:
    _json.dump(bb_ce_results, f)
print("✅ BanglaBERT + CrossEntropy results saved")

encoder_effect = bb_ce_results["eval_f1_macro"] - mbert_test_results["eval_f1_macro"]
print(f"\n📊 Encoder effect (BanglaBERT-CE minus mBERT-CE): {encoder_effect:+.4f}")

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/3216 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

✅ Baseline 2 — BanglaBERT + CrossEntropy | Test Macro F1: 0.6298
✅ BanglaBERT + CrossEntropy results saved

📊 Encoder effect (BanglaBERT-CE minus mBERT-CE): +0.0023


In [11]:
#CELL 9 — Proposed: BanglaBERT + Focal Loss
#============================================================
model_bb = AutoModelForTokenClassification.from_pretrained(
    "csebuetnlp/banglabert",
    num_labels=len(label_list), id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)

trainer_bb = FocalLossTrainer(
    focal_loss_fn=focal_loss_fn,
    model=model_bb,
    args=TrainingArguments(
        output_dir="/kaggle/working/outputs_ner",
        num_train_epochs            = 10,   
        per_device_train_batch_size = 16,   
        learning_rate               = 2e-5, 
        weight_decay                = 0.01, 
        max_grad_norm               = 1.0,  
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        seed=42, report_to="none",
    ),
    train_dataset=tok_bb["train"],
    eval_dataset=tok_bb["validation"],
    processing_class=tokenizer_bb,
    data_collator=collator_bb,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_bb.train()
print("✅ Proposed — BanglaBERT + Focal Loss training done")

# Full per-class test evaluation
out = trainer_bb.predict(tok_bb["test"])
preds_bb = np.argmax(out.predictions, axis=2)
true_labels_bb = [[id2label[l] for l in lab if l!=-100] for lab in out.label_ids]
true_preds_bb  = [[id2label[p] for p,l in zip(pr,lab) if l!=-100]
                  for pr,lab in zip(preds_bb, out.label_ids)]

bb_focal_macro = f1_score(true_labels_bb, true_preds_bb, average="macro")
bb_focal_report = classification_report(true_labels_bb, true_preds_bb, output_dict=True)

print(f"\n✅ Proposed — BanglaBERT + Focal Loss | Test Macro F1: {bb_focal_macro:.4f}")
print(f"\n📊 Optimization strategy effect (Proposed minus BanglaBERT-CE): {bb_focal_macro - bb_ce_results['eval_f1_macro']:+.4f}")
print(f"\n=== PER-CLASS RESULTS ===")
print(classification_report(true_labels_bb, true_preds_bb))

# Save per-class results for Notebook 3
bb_focal_results = {
    "eval_f1_macro":        bb_focal_macro,
    "eval_precision_macro": precision_score(true_labels_bb, true_preds_bb, average="macro"),
    "eval_recall_macro":    recall_score(true_labels_bb, true_preds_bb, average="macro"),
    "per_class":            bb_focal_report,
}
def convert(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    raise TypeError

with open("/kaggle/working/bb_focal_ner_results.json", "w") as f:
    _json.dump(bb_focal_results, f, ensure_ascii=False, indent=2, default=convert)
print("✅ BanglaBERT + Focal results saved → bb_focal_ner_results.json")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro
1,No log,0.577180,0.052107,0.078704,0.038946
2,No log,0.378423,0.448314,0.480881,0.492208
3,No log,0.288870,0.608144,0.609735,0.638888
4,No log,0.274248,0.642531,0.588646,0.718331
5,0.766602,0.237865,0.685714,0.657893,0.718122
6,0.766602,0.229503,0.689193,0.652625,0.734467
7,0.766602,0.225871,0.669517,0.624470,0.727207
8,0.766602,0.222205,0.701446,0.657113,0.755481
9,0.766602,0.224546,0.699510,0.648064,0.760760
10,0.155784,0.224287,0.701120,0.649310,0.765762


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

✅ Proposed — BanglaBERT + Focal Loss training done



✅ Proposed — BanglaBERT + Focal Loss | Test Macro F1: 0.6725

📊 Optimization strategy effect (Proposed minus BanglaBERT-CE): +0.0427

=== PER-CLASS RESULTS ===
              precision    recall  f1-score   support

         ANI       0.50      0.33      0.40        12
         COL       0.80      0.94      0.86        17
        FOOD       0.86      0.86      0.86        49
         LOC       0.65      0.80      0.71        44
         OBJ       0.59      0.80      0.68        51
         ORG       0.85      0.68      0.76        25
         PER       0.62      0.71      0.67         7
         REL       0.70      0.80      0.75       112
        ROLE       0.27      0.55      0.36        11

   micro avg       0.68      0.78      0.72       328
   macro avg       0.65      0.72      0.67       328
weighted avg       0.69      0.78      0.73       328

✅ BanglaBERT + Focal results saved → bb_focal_ner_results.json


In [12]:
#CELL 10 — Per-Dialect NER Evaluation (Proposed model)
#============================================================
syl_test  = [r for r in raw_test_rows if r["dialect"]=="Sylheti"]
chit_test = [r for r in raw_test_rows if r["dialect"]=="Chittagonian"]

for dialect, rows in [("Sylheti", syl_test), ("Chittagonian", chit_test)]:
    tok_d = build_tokenized_datasets(tokenizer_bb, downsampled_train, raw_val_rows, rows)
    out_d = trainer_bb.predict(tok_d["test"])
    preds_d = np.argmax(out_d.predictions, axis=2)
    tl = [[id2label[l] for l in lab if l!=-100] for lab in out_d.label_ids]
    tp = [[id2label[p] for p,l in zip(pr,lab) if l!=-100]
          for pr,lab in zip(preds_d, out_d.label_ids)]
    f1_d = f1_score(tl, tp, average="macro")
    print(f"\n=== {dialect} (n={len(rows)}) ===")
    print(classification_report(tl, tp))
    print(f"Macro F1 ({dialect}): {f1_d:.4f}")

Map:   0%|          | 0/3216 [00:00<?, ? examples/s]

Map:   0%|          | 0/671 [00:00<?, ? examples/s]

Map:   0%|          | 0/333 [00:00<?, ? examples/s]


=== Sylheti (n=333) ===
              precision    recall  f1-score   support

         ANI       1.00      1.00      1.00         1
         COL       0.89      0.89      0.89         9
        FOOD       0.88      0.96      0.92        23
         LOC       0.63      0.74      0.68        23
         OBJ       0.68      0.81      0.74        21
         ORG       0.79      0.85      0.81        13
         PER       1.00      0.67      0.80         3
         REL       0.62      0.79      0.69        52
        ROLE       0.38      0.56      0.45         9

   micro avg       0.68      0.81      0.74       154
   macro avg       0.76      0.81      0.78       154
weighted avg       0.69      0.81      0.74       154

Macro F1 (Sylheti): 0.7766


Map:   0%|          | 0/3216 [00:00<?, ? examples/s]

Map:   0%|          | 0/671 [00:00<?, ? examples/s]

Map:   0%|          | 0/340 [00:00<?, ? examples/s]


=== Chittagonian (n=340) ===
              precision    recall  f1-score   support

         ANI       0.43      0.27      0.33        11
         COL       0.73      1.00      0.84         8
        FOOD       0.83      0.77      0.80        26
         LOC       0.67      0.86      0.75        21
         OBJ       0.55      0.80      0.65        30
         ORG       1.00      0.50      0.67        12
         PER       0.50      0.75      0.60         4
         REL       0.78      0.82      0.80        60
        ROLE       0.11      0.50      0.18         2

   micro avg       0.67      0.76      0.71       174
   macro avg       0.62      0.70      0.62       174
weighted avg       0.71      0.76      0.72       174

Macro F1 (Chittagonian): 0.6244


In [14]:
# CELL 10a_cleanup — Clear failed transfer checkpoints
import shutil, os, subprocess

for folder in [
    "/kaggle/working/transfer_Syl_to_Chit",
    "/kaggle/working/transfer_Chit_to_Syl",
]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"🗑️ Deleted: {folder}")

result = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print(result.stdout)

🗑️ Deleted: /kaggle/working/transfer_Syl_to_Chit
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   16G  4.2G  79% /kaggle/working



In [16]:
# ============================================================
# CELL 10b — Cross-Dialect Transfer Experiment (UPDATED)
# ============================================================
import random
import numpy as np
import torch
from transformers import AutoModelForTokenClassification, TrainingArguments, EarlyStoppingCallback
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# -----------------------------
# Transfer function
# -----------------------------
def run_transfer_experiment(train_rows, test_rows, exp_label):
    tok_t = build_tokenized_datasets(
        tokenizer_bb,
        train_rows,
        raw_val_rows,
        test_rows
    )

    model = AutoModelForTokenClassification.from_pretrained(
        "csebuetnlp/banglabert",
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    trainer = FocalLossTrainer(
        focal_loss_fn=focal_loss_fn,
        model=model,
        args=TrainingArguments(
            output_dir=f"/kaggle/working/transfer_{exp_label}",
            num_train_epochs=10,
            per_device_train_batch_size=16,
            learning_rate=2e-5,
            weight_decay=0.01,
            max_grad_norm=1.0,
            eval_strategy="epoch",
            save_strategy="epoch",                  # ← changed
            save_only_model=True,                   # ← added
            load_best_model_at_end=True,            # ← added
            metric_for_best_model="eval_f1_macro",  # ← added
            greater_is_better=True,                 # ← added
            seed=42,
            report_to="none",
        ),
        train_dataset=tok_t["train"],
        eval_dataset=tok_t["validation"],
        processing_class=tokenizer_bb,
        data_collator=collator_bb,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # ← added
    )

    trainer.train()

    out = trainer.predict(tok_t["test"])
    preds = np.argmax(out.predictions, axis=2)

    true_labels = [
        [id2label[l] for l in lab if l != -100]
        for lab in out.label_ids
    ]

    true_preds = [
        [id2label[p] for p, l in zip(pr, lab) if l != -100]
        for pr, lab in zip(preds, out.label_ids)
    ]

    f1 = f1_score(true_labels, true_preds)
    p = precision_score(true_labels, true_preds)
    r = recall_score(true_labels, true_preds)

    print("\n=== Transfer:", exp_label, "===")
    print(classification_report(true_labels, true_preds))
    print(f"Macro F1 ({exp_label}): {f1:.4f}")

    return f1

# -----------------------------
# Prepare dialect splits
# -----------------------------
syl_train_rows  = [r for r in downsampled_train if r["dialect"] == "Sylheti"]
chit_train_rows = [r for r in downsampled_train if r["dialect"] == "Chittagonian"]

print("\n🔁 Running cross-dialect transfer experiments...")

f1_syl_to_chit = run_transfer_experiment(
    syl_train_rows,
    chit_test,
    "Syl_to_Chit"
)

f1_chit_to_syl = run_transfer_experiment(
    chit_train_rows,
    syl_test,
    "Chit_to_Syl"
)

# -----------------------------
# Summary
# -----------------------------
print("\n📊 CROSS-DIALECT TRANSFER SUMMARY")
print(f"   Syl→Chit Macro F1:  {f1_syl_to_chit:.4f}")
print(f"   Chit→Syl Macro F1:  {f1_chit_to_syl:.4f}")
print(f"   In-domain Macro F1: {bb_focal_macro:.4f}")
print(f"   Dialect gap (avg):  {bb_focal_macro - (f1_syl_to_chit + f1_chit_to_syl)/2:.4f}")

transfer_results = {
    "syl_to_chit": f1_syl_to_chit,
    "chit_to_syl": f1_chit_to_syl,
    "in_domain": bb_focal_macro,
}

print("✅ transfer_results ready")


🔁 Running cross-dialect transfer experiments...


Map:   0%|          | 0/1585 [00:00<?, ? examples/s]

Map:   0%|          | 0/671 [00:00<?, ? examples/s]

Map:   0%|          | 0/340 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye


=== Transfer: Chit_to_Syl ===
              precision    recall  f1-score   support

         ANI       0.00      0.00      0.00         1
         COL       0.82      1.00      0.90         9
        FOOD       0.78      0.78      0.78        23
         LOC       0.45      0.65      0.54        23
         OBJ       0.47      0.67      0.55        21
         ORG       0.86      0.46      0.60        13
         PER       1.00      0.33      0.50         3
         REL       0.49      0.81      0.61        52
        ROLE       0.43      0.33      0.38         9

   micro avg       0.55      0.70      0.62       154
   macro avg       0.59      0.56      0.54       154
weighted avg       0.58      0.70      0.61       154

Macro F1 (Chit_to_Syl): 0.6154

📊 CROSS-DIALECT TRANSFER SUMMARY
   Syl→Chit Macro F1:  0.4921
   Chit→Syl Macro F1:  0.6154
   In-domain Macro F1: 0.6725
   Dialect gap (avg):  0.1187
✅ transfer_results ready


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [17]:
# ── Mixed training (AFTER transfer) ────────────────────
print("\n🔁 Mixed training experiment...")

from transformers import EarlyStoppingCallback, AutoModelForTokenClassification, TrainingArguments
from seqeval.metrics import f1_score as seq_f1  # ✅ explicit alias, sklearn নেই
import numpy as np
import json as _json

# Dataset
tok_mixed = build_tokenized_datasets(
    tokenizer_bb,
    downsampled_train,
    raw_val_rows,
    raw_test_rows
)

# Model
m_mixed = AutoModelForTokenClassification.from_pretrained(
    "csebuetnlp/banglabert",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# Training Args — Cell 9 (Proposed) এর সাথে IDENTICAL
training_args = TrainingArguments(
    output_dir="/kaggle/working/transfer_mixed",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="linear",          # ✅ added
    warmup_ratio=0.1,                    # ✅ added
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,                # ✅ added
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",  # ✅ fixed (was eval_loss)
    greater_is_better=True,              # ✅ added
    seed=42,
    report_to="none",
)

# Trainer
t_mixed = FocalLossTrainer(
    focal_loss_fn=focal_loss_fn,
    model=m_mixed,
    args=training_args,
    train_dataset=tok_mixed["train"],
    eval_dataset=tok_mixed["validation"],
    processing_class=tokenizer_bb,
    data_collator=collator_bb,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

t_mixed.train()
print("✅ Mixed training done")

# Evaluation
tok_syl  = build_tokenized_datasets(tokenizer_bb, downsampled_train, raw_val_rows, syl_test)
tok_chit = build_tokenized_datasets(tokenizer_bb, downsampled_train, raw_val_rows, chit_test)
tok_comb = build_tokenized_datasets(tokenizer_bb, downsampled_train, raw_val_rows, raw_test_rows)

test_sets = {
    "Sylheti":      tok_syl,
    "Chittagonian": tok_chit,
    "Combined":     tok_comb,
}

mixed_results = {}

for dialect, tok_d in test_sets.items():
    out_d   = t_mixed.predict(tok_d["test"])
    preds_d = np.argmax(out_d.predictions, axis=2)

    tl = [[id2label[l] for l in lab if l != -100] for lab in out_d.label_ids]
    tp = [[id2label[p] for p, l in zip(pr, lab) if l != -100]
          for pr, lab in zip(preds_d, out_d.label_ids)]

    f1_d = seq_f1(tl, tp, average="macro")  # ✅ seqeval
    mixed_results[dialect] = f1_d
    print(f"Mixed → {dialect}: Macro F1 = {f1_d:.4f}")

# Save
transfer_results["mixed_sylheti"]      = mixed_results["Sylheti"]
transfer_results["mixed_chittagonian"] = mixed_results["Chittagonian"]
transfer_results["mixed_combined"]     = mixed_results["Combined"]

with open("/kaggle/working/transfer_results.json", "w") as f:
    _json.dump(transfer_results, f, indent=2, ensure_ascii=False)

print("✅ Transfer + Mixed results saved")


🔁 Mixed training experiment...


Map:   0%|          | 0/3216 [00:00<?, ? examples/s]

Map:   0%|          | 0/671 [00:00<?, ? examples/s]

Map:   0%|          | 0/673 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Map:   0%|          | 0/671 [00:00<?, ? examples/s]

Map:   0%|          | 0/673 [00:00<?, ? examples/s]

Mixed → Sylheti: Macro F1 = 0.7766


Mixed → Chittagonian: Macro F1 = 0.6244


Mixed → Combined: Macro F1 = 0.6725
✅ Transfer + Mixed results saved


In [18]:
# CELL 10c — Disk Cleanup before push
import shutil, os, subprocess

for folder in [
    "/kaggle/working/outputs_mbert",
    "/kaggle/working/outputs_ner_bb_ce", 
    "/kaggle/working/outputs_ner",
    "/kaggle/working/transfer_mixed",
    "/kaggle/working/transfer_Syl_to_Chit",
    "/kaggle/working/transfer_Chit_to_Syl",
]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"🗑️ Deleted: {folder}")

result = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print(result.stdout)

🗑️ Deleted: /kaggle/working/transfer_mixed
🗑️ Deleted: /kaggle/working/transfer_Syl_to_Chit
🗑️ Deleted: /kaggle/working/transfer_Chit_to_Syl
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  561M   19G   3% /kaggle/working



In [19]:
# ============================================================
# CELL 11 — Push All Models to HuggingFace
# ============================================================

# Baseline 1: mBERT + CrossEntropy
trainer_mbert.model.push_to_hub("ashik297/mbert-bangla-dialect-ner")
tokenizer_mbert.push_to_hub("ashik297/mbert-bangla-dialect-ner")
print("✅ Baseline 1 pushed: ashik297/mbert-bangla-dialect-ner")

# Baseline 2: BanglaBERT + CrossEntropy
trainer_bb_ce.model.push_to_hub("ashik297/banglabert-ce-bangla-dialect-ner")
tokenizer_bb.push_to_hub("ashik297/banglabert-ce-bangla-dialect-ner")
print("✅ Baseline 2 pushed: ashik297/banglabert-ce-bangla-dialect-ner")

# Proposed: BanglaBERT + Focal Loss
trainer_bb.model.push_to_hub("ashik297/banglabert-bangla-dialect-ner")
tokenizer_bb.push_to_hub("ashik297/banglabert-bangla-dialect-ner")
print("✅ Proposed pushed: ashik297/banglabert-bangla-dialect-ner")

# Mixed: BanglaBERT + Focal Loss + Mixed
t_mixed.model.push_to_hub("ashik297/banglabert-mixed-bangla-dialect-ner")
tokenizer_bb.push_to_hub("ashik297/banglabert-mixed-bangla-dialect-ner")
print("✅ Mixed pushed: ashik297/banglabert-mixed-bangla-dialect-ner")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

✅ Baseline 1 pushed: ashik297/mbert-bangla-dialect-ner


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

✅ Mixed pushed: ashik297/banglabert-mixed-bangla-dialect-ner


In [20]:
#CELL 12 — Final Summary
#============================================================
print("\n" + "="*65)
print("NER TRAINING COMPLETE — FINAL RESULTS")
print("="*65)
print(f"{'Model':<30} {'F1':>8} {'P':>8} {'R':>8}")
print("-"*65)
print(f"{'Baseline 1: mBERT + CE':<30} {mbert_test_results['eval_f1_macro']:>8.4f} {mbert_test_results['eval_precision_macro']:>8.4f} {mbert_test_results['eval_recall_macro']:>8.4f}")
print(f"{'Baseline 2: BanglaBERT + CE':<30} {bb_ce_results['eval_f1_macro']:>8.4f} {bb_ce_results['eval_precision_macro']:>8.4f} {bb_ce_results['eval_recall_macro']:>8.4f}")
print(f"{'Proposed: BanglaBERT + Focal':<30} {bb_focal_macro:>8.4f} {bb_focal_results['eval_precision_macro']:>8.4f} {bb_focal_results['eval_recall_macro']:>8.4f}")
print("-"*65)
print(f"  Encoder effect (F1):              {encoder_effect:+.4f}")
print(f"  Optimization strategy effect (F1): {bb_focal_macro - bb_ce_results['eval_f1_macro']:+.4f}")
print(f"\n  Transfer — Syl→Chit: {f1_syl_to_chit:.4f} | Chit→Syl: {f1_chit_to_syl:.4f}")
print(f"  Dialect gap: {bb_focal_macro - (f1_syl_to_chit+f1_chit_to_syl)/2:.4f}")
print("\n⚠️  PER (test n=7) and ANI (test n=12): sparse — results included in macro, interpret cautiously (noted in paper Limitations)")
print("\n✅ Download before session ends:")
print("   - mbert_ner_results.json")
print("   - bb_ce_ner_results.json")
print("   - bb_focal_ner_results.json")
print("   - transfer_results.json")


NER TRAINING COMPLETE — FINAL RESULTS
Model                                F1        P        R
-----------------------------------------------------------------
Baseline 1: mBERT + CE           0.6275   0.6265   0.6644
Baseline 2: BanglaBERT + CE      0.6298   0.6345   0.6611
Proposed: BanglaBERT + Focal     0.6725   0.6494   0.7194
-----------------------------------------------------------------
  Encoder effect (F1):              +0.0023
  Optimization strategy effect (F1): +0.0427

  Transfer — Syl→Chit: 0.4921 | Chit→Syl: 0.6154
  Dialect gap: 0.1187

⚠️  PER (test n=7) and ANI (test n=12): sparse — results included in macro, interpret cautiously (noted in paper Limitations)

✅ Download before session ends:
   - mbert_ner_results.json
   - bb_ce_ner_results.json
   - bb_focal_ner_results.json
   - transfer_results.json


In [21]:
# All models per-dialect breakdown + save
import json
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_p, recall_score as seq_r

all_dialect_results = {}

for model_name, trainer in [
    ("mBERT_CE", trainer_mbert),
    ("BanglaBERT_CE", trainer_bb_ce),
    ("BanglaBERT_Focal", trainer_bb),
    ("BanglaBERT_Mixed", t_mixed),
]:
    all_dialect_results[model_name] = {}
    
    for dialect, rows in [("Sylheti", syl_test), ("Chittagonian", chit_test)]:
        tok_d = build_tokenized_datasets(
            tokenizer_mbert if model_name == "mBERT_CE" else tokenizer_bb,
            downsampled_train, raw_val_rows, rows
        )
        out_d = trainer.predict(tok_d["test"])
        preds_d = np.argmax(out_d.predictions, axis=2)
        tl = [[id2label[l] for l in lab if l != -100] for lab in out_d.label_ids]
        tp = [[id2label[p] for p, l in zip(pr, lab) if l != -100]
              for pr, lab in zip(preds_d, out_d.label_ids)]
        
        all_dialect_results[model_name][dialect] = {
            "f1_macro":        seq_f1(tl, tp, average="macro"),
            "precision_macro": seq_p(tl, tp, average="macro"),
            "recall_macro":    seq_r(tl, tp, average="macro"),
        }
        print(f"{model_name} → {dialect}: F1={all_dialect_results[model_name][dialect]['f1_macro']:.4f}")

with open("/kaggle/working/all_models_per_dialect_results.json", "w") as f:
    json.dump(all_dialect_results, f, indent=2, ensure_ascii=False)

print("✅ Saved: all_models_per_dialect_results.json")

Map:   0%|          | 0/3216 [00:00<?, ? examples/s]

Map:   0%|          | 0/671 [00:00<?, ? examples/s]

Map:   0%|          | 0/333 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


mBERT_CE → Sylheti: F1=0.7262


...

BanglaBERT_Mixed → Sylheti: F1=0.7766


Map:   0%|          | 0/3216 [00:00<?, ? examples/s]

Map:   0%|          | 0/671 [00:00<?, ? examples/s]

Map:   0%|          | 0/340 [00:00<?, ? examples/s]

BanglaBERT_Mixed → Chittagonian: F1=0.6244
✅ Saved: all_models_per_dialect_results.json
